In [1]:
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords

In [2]:
df = pd.read_csv("../movies_data.csv")
df.sample(10)

,review,sentiment
43008,This movie will not be considered for an acade...,1
21264,This was the second Cinemascope spectacle that...,1
3987,"Well, i must admit, when i saw the trailer for...",0
32438,**SPOILERS** Simple movie about simple people ...,1
30042,"The plot of ""In the Mood for Love"" is simple e...",1
40790,i would have given this movie a 1 out of 10 if...,0
2150,"This movie is more Lupin then most, especially...",1
49495,A brilliant professor and his sidekick journey...,0
16447,"""Secret Sunshine"" reminded me of ""The Rapture""...",1
13513,It has singing. It has drama. It has comedy. I...,1


In [3]:
stop = stopwords.words('english')

def tokenizer(text):
    text = re.sub('<[^>]*>','',text)
    emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)',text)
    text = (re.sub('[\W]+', ' ', text.lower()) + ' '.join(emoticons).replace('-', ''))
    tokenized = [word for word in text.split() if word not in stop]
    return tokenized

In [4]:
def stream_docs(path):
    with open(path, 'r', encoding='utf-8') as csv:
        next(csv)
        for line in csv:
            review, sentiment = line[:-3], int(line[-2])
            yield review, sentiment

next(stream_docs("../movies_data.csv"))

('"Any movie that portrays the hard-working responsible husband as the person who has to change because of bored, cheating wife is an obvious result of 8 years of the Clinton era.<br /><br />It\'s little wonder that this movie was written by a woman."',
 0)

In [5]:
def get_mini_batch(doc_stream, size):
    docs, y = [], []
    try:
        for _ in range(size):
            review, sentiment = next(doc_stream)
            docs.append(review)
            y.append(sentiment)
    except StopIteration:
        return None, None
    return docs, y

In [6]:
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier

vect = HashingVectorizer(decode_error='ignore',
                        n_features=2**21,
                        preprocessor=None,
                        tokenizer=tokenizer)
clf = SGDClassifier(loss='log_loss',random_state=14)
doc_stream = stream_docs("../movies_data.csv")

In [7]:
import pyprind
pbar = pyprind.ProgBar(45)
classes = np.array([0,1])

for _ in range(45):
    X_train, y_train = get_mini_batch(doc_stream, size=1000)
    if not X_train:
        break
    X_train = vect.transform(X_train)
    clf.partial_fit(X_train, y_train, classes=classes)
    pbar.update()

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:23


In [8]:
X_test, y_test = get_mini_batch(doc_stream, size=5000)
X_test = vect.transform(X_test)
print(f"Accuracy: {clf.score(X_test, y_test):.3f}")

Accuracy: 0.872
